# Simple Therapy Flattening Figure for BD&S

Creates a clear, interpretable figure showing selective (human) vs uniform (LLM) therapy language deployment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Publication settings (per CLAUDE.md)
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

FIGURES_DIR = Path('../figures')

In [ ]:
# Load data
df = pd.read_parquet('../data/advice_metrics.parquet')
topics_df = pd.read_parquet('../data/multi_model_assignments.parquet')[['post_id', 'majority_topic']]
topics_df = topics_df.rename(columns={'majority_topic': 'primary_topic'})

df_topics = df.merge(topics_df, on='post_id', how='inner')
print(f"Rows: {len(df_topics)}")
print(f"Sources: {df_topics['source'].unique()}")

In [ ]:
# Normalize therapy count per 100 tokens
df_topics['therapy_per100'] = df_topics['therapy_count'] / df_topics['n_tokens'] * 100

# Compute per-topic MEAN (not median - median is 0 for 83% of human responses)
therapy_by_topic = df_topics.groupby(['primary_topic', 'source'])['therapy_per100'].mean().unstack('source')

# Only topics with enough data
topic_counts = df_topics.groupby('primary_topic').size()
valid_topics = topic_counts[topic_counts >= 50].index
therapy_by_topic = therapy_by_topic.loc[therapy_by_topic.index.isin(valid_topics)]

# Compute LLM average
llm_cols = ['gemini', 'deepseek', 'ministral', 'gpt_nano']
llm_cols = [c for c in llm_cols if c in therapy_by_topic.columns]
therapy_by_topic['llm_avg'] = therapy_by_topic[llm_cols].mean(axis=1)

print(f"Topics with 50+ posts: {len(therapy_by_topic)}")
print("\nHuman therapy rate range:")
print(f"  Min: {therapy_by_topic['human'].min():.3f}")
print(f"  Max: {therapy_by_topic['human'].max():.3f}")
print(f"  Mean: {therapy_by_topic['human'].mean():.3f}")
therapy_by_topic[['human', 'llm_avg']].head(10)

In [ ]:
# Coefficient of variation
human_cv = therapy_by_topic['human'].std() / therapy_by_topic['human'].mean()
llm_cv = therapy_by_topic['llm_avg'].std() / therapy_by_topic['llm_avg'].mean()
print(f"Human CV: {human_cv:.2f}")
print(f"LLM CV: {llm_cv:.2f}")

In [ ]:
# Select 8 topics that illustrate the contrast
# We want: some where human is LOW (practical), some where human is HIGH (emotional)
# And show that LLM is high on ALL of them

# Sort by human rate
sorted_topics = therapy_by_topic.sort_values('human')

# Pick 4 from bottom (low human therapy) and 4 from top (high human therapy)
low_human = sorted_topics.head(15)  # Candidates for low human therapy
high_human = sorted_topics.tail(15)  # Candidates for high human therapy

print("LOW human therapy topics:")
print(low_human[['human', 'llm_avg']].round(3))
print("\nHIGH human therapy topics:")
print(high_human[['human', 'llm_avg']].round(3))

In [ ]:
# Data-driven topic selection
# Pick topics that actually show the contrast in the data

# Sort by human therapy rate
sorted_topics = therapy_by_topic.sort_values('human')

print("All topics sorted by human therapy rate:")
print(sorted_topics[['human', 'llm_avg']].round(3).to_string())

# Select 4 topics with LOWEST human therapy (where gap is largest)
# and 4 topics with HIGHEST human therapy (where both are elevated)
low_4 = sorted_topics.head(4).index.tolist()
high_4 = sorted_topics.tail(4).index.tolist()

selected_topics = low_4 + high_4
print(f"\nSelected LOW human: {low_4}")
print(f"Selected HIGH human: {high_4}")

In [ ]:
# Get data for selected topics
plot_data = therapy_by_topic.loc[selected_topics][['human', 'llm_avg']].copy()

# Sort by human rate for visual clarity
plot_data = plot_data.sort_values('human')

# Clean up topic names for display (generic approach)
def clean_topic_name(name):
    # Replace underscores, title case, wrap long names
    clean = name.replace('_', ' ').title()
    # Wrap at ~15 chars
    if len(clean) > 18:
        words = clean.split()
        mid = len(words) // 2
        clean = ' '.join(words[:mid]) + '\n' + ' '.join(words[mid:])
    return clean

plot_data.index = [clean_topic_name(t) for t in plot_data.index]

print("Data for plot:")
print(plot_data.round(3))

In [ ]:
# Dumbbell chart - horizontal, sorted by human value
# Shows the gap clearly without implying false continuity

fig, ax = plt.subplots(figsize=(6, 4.5), dpi=150)

# Sort by human therapy density
plot_sorted = plot_data.sort_values('human')
y = np.arange(len(plot_sorted))

# Draw connecting lines (the "dumbbells")
for i, (topic, row) in enumerate(plot_sorted.iterrows()):
    ax.plot([row['human'], row['llm_avg']], [i, i], 
            color='#cccccc', linewidth=2, zorder=1)

# Draw dots
ax.scatter(plot_sorted['human'], y, color='#000000', s=80, 
           label='Human', zorder=10, edgecolors='white', linewidths=0.5)
ax.scatter(plot_sorted['llm_avg'], y, color='#666666', s=80, marker='s',
           label='LLM (average)', zorder=10, edgecolors='white', linewidths=0.5)

# Formatting
ax.set_xlabel('Therapy terms per 100 tokens', fontweight='bold')
ax.set_yticks(y)
ax.set_yticklabels(plot_sorted.index, fontsize=9)

ax.legend(loc='lower right', framealpha=0.9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.grid(axis='x', alpha=0.3, linestyle='--')

# X-axis with padding so circles at 0 aren't cut off
x_max = max(plot_sorted['llm_avg'].max(), plot_sorted['human'].max()) * 1.15
ax.set_xlim(-0.03 * x_max, x_max)

# Light horizontal lines for readability
for i in y:
    ax.axhline(y=i, color='#eeeeee', linewidth=0.5, zorder=0)

plt.tight_layout()
plt.show()

In [ ]:
# Save figure
fig.savefig(FIGURES_DIR / 'fig_therapy_flattening_simple.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURES_DIR / 'fig_therapy_flattening_simple.png', bbox_inches='tight', dpi=300)
print('Saved: fig_therapy_flattening_simple.pdf/png')

In [ ]:
# Print stats for caption
print(f"\nFor caption:")
print(f"Human CV: {human_cv:.2f}")
print(f"LLM CV: {llm_cv:.2f}")
print(f"\nHuman range: {plot_data['human'].min():.2f} to {plot_data['human'].max():.2f}")
print(f"LLM range: {plot_data['llm_avg'].min():.2f} to {plot_data['llm_avg'].max():.2f}")